# Автопризнаки

## Настройки

In [1]:
%pip -q install "numpy>=1.24" "numba>=0.59" "pyarrow>=16" "catboost>=1.2" "huggingface_hub>=0.30" tqdm

In [26]:
import gc, json, time, shutil, os, hashlib, ctypes, psutil
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from numba import njit
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from huggingface_hub import hf_hub_download
from sklearn.metrics import roc_auc_score, log_loss
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TLab_Yambda')
DATA = ROOT / 'retrieval_gts_v2'
R2 = ROOT / 'catboost_reranking_v1'
OUT = ROOT / 'adaptive_bins_autofe_sparse_v3'
LOCAL = Path('/content/yambda_adaptive_bins_sparse_v3')
OUT.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

SEED = 42
C = 600
DAY = 86_400
GAP = 1_800
N_COARSE = 6
N_CHANNELS = 10
Z_DIM = 64
HIST_DAYS = 10
HIST_POS_PER_USER = 8
NEG = 12
BATCH_GROUPS = 384
ENC_EPOCHS = 16
ENC_LR = 5e-4
ENC_PATIENCE = 4
CB_NEG_PER_USER = 100
CB_EVAL_USERS = 512
CB_TREES, CB_DEPTH, CB_LR = 500, 8, 0.05
INFER_BATCH = 4096
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RAM_SHARE = 0.48
RAM_MAX_GIB = 3.0


FINE_EDGES = np.r_[np.expm1(np.linspace(0, np.log1p(365), 24)), 1e8].astype(np.float32)
NF = len(FINE_EDGES)-1
FINE_CENTERS = np.r_[(np.log1p(FINE_EDGES[:-2]) + np.log1p(FINE_EDGES[1:-1]))/2,
                     np.log1p(730.)].astype(np.float32)

rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
users = pd.read_csv(DATA/'selected_users.csv').uid.to_numpy(dtype=np.uint32)
N_USERS = len(users)
snaps = pd.read_csv(DATA/'snapshots.csv').set_index('split')
print('Device:',DEVICE,'users:',N_USERS,'technical bins:',NF)
display(snaps)

config = dict(storage='sparse_bins_v3',seed=SEED, fine_edges_days=FINE_EDGES.tolist(), n_coarse=N_COARSE,
              n_channels=N_CHANNELS, z_dim=Z_DIM, hist_days=HIST_DAYS,
              hist_pos_per_user=HIST_POS_PER_USER, neg=NEG, batch_groups=BATCH_GROUPS,
              encoder_epochs=ENC_EPOCHS, encoder_lr=ENC_LR, gap_seconds=GAP)
(OUT/'config.json').write_text(json.dumps(config,indent=2))

import subprocess, sys
with (OUT/'requirements-lock.txt').open('w') as lockfile:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=lockfile, check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda users: 10000 technical bins: 24


,snapshot_id,cutoff,history_end,end
split,,,,
train,0,25737200,25735400,25823600
valid,1,25825400,25823600,25911800
test,2,25913600,25911800,26000000


834

## История

In [3]:
cols = ['uid','item_id','timestamp','event_type','played_ratio_pct',
        'is_organic','track_length_seconds','positive']
events = pd.read_parquet(DATA/'events_sample.parquet', columns=cols)
events = events.sort_values('timestamp',kind='stable').reset_index(drop=True)
EU = pd.Index(users).get_indexer(events.uid).astype(np.int32)
EI_RAW = events.item_id.to_numpy(dtype=np.uint32)
ET = events.timestamp.to_numpy(dtype=np.int64)
ER = np.nan_to_num(events.played_ratio_pct.to_numpy(dtype=np.float32),nan=0.)/100
EO = events.is_organic.fillna(False).to_numpy(dtype=np.uint8)
EL = np.nan_to_num(events.track_length_seconds.to_numpy(dtype=np.float32),nan=0.)
EK = np.full(len(events),6,dtype=np.uint8)
kind = events.event_type.astype(str)
for label,idx in [('listen',0),('like',2),('unlike',3),('dislike',4),('undislike',5)]:
    EK[kind.eq(label).to_numpy()] = idx
EK[(EK==0)&events.positive.to_numpy(dtype=bool)] = 1
items = np.unique(EI_RAW)
EI = np.searchsorted(items,EI_RAW).astype(np.int32)
N_ITEMS = len(items)
del events,kind

artist_file = hf_hub_download(repo_id='yandex/yambda',repo_type='dataset',
                               filename='artist_item_mapping.parquet')
art = pd.read_parquet(artist_file,columns=['item_id','artist_id']).dropna().drop_duplicates()
art = art[art.item_id.isin(items)].sort_values(['item_id','artist_id'])
ai = np.searchsorted(items,art.item_id.to_numpy(dtype=np.uint32))
acount = np.bincount(ai,minlength=N_ITEMS)
APTR = np.r_[0,np.cumsum(acount,dtype=np.int64)]
ART = art.artist_id.to_numpy(dtype=np.uint32)
del art,ai,acount
print('Events:',f'{len(ET):,}','Items:',f'{N_ITEMS:,}','Artist links:',f'{len(ART):,}')


Events: 47,790,449 Items: 934,057 Artist links: 928,796


## Агрегаты

In [4]:
@njit
def locate(keys,value):
    lo,hi = 0,len(keys)
    while lo<hi:
        mid=(lo+hi)//2
        if keys[mid]<value: lo=mid+1
        else: hi=mid
    return lo if lo<len(keys) and keys[lo]==value else -1

@njit
def mark_relations(u,raw,item_idx,artist_ptr,artist_ids,ui_keys,ua_keys):
    seen_ui=np.zeros(len(ui_keys),np.uint8)
    seen_ua=np.zeros(len(ua_keys),np.uint8)
    for e in range(len(u)):
        prefix=np.uint64(u[e])<<np.uint64(32)
        p=locate(ui_keys,prefix|np.uint64(raw[e]))
        if p>=0: seen_ui[p]=1
        it=item_idx[e]
        for a in range(artist_ptr[it],artist_ptr[it+1]):
            q=locate(ua_keys,prefix|np.uint64(artist_ids[a]))
            if q>=0: seen_ua[q]=1
    return seen_ui,seen_ua

@njit
def put(state,row,b,event,played,organic,length):
    state[row,b,event]+=1.
    if event<2:
        state[row,b,7]+=played
        state[row,b,8]+=organic
        state[row,b,9]+=length/300.

@njit
def aggregate_history_into(U,I,UI,UA,last_u,last_i,last_ui,first_ui,
                           last_ratio,like,dislike,last_ua,
                           u,raw,item_idx,ts,kind,ratio,organic,length,
                           hist_end,edges,ptr,artists,ui_keys,ua_keys,ui_map,ua_map):

    for e in range(len(u)):
        age=(hist_end-ts[e])/86400.
        b=0
        while b<len(edges)-2 and age>=edges[b+1]: b+=1
        uid,it,event=u[e],item_idx[e],kind[e]
        put(U,uid,b,event,ratio[e],organic[e],length[e])
        put(I,it,b,event,ratio[e],organic[e],length[e])
        last_u[uid]=ts[e]
        last_i[it]=ts[e]
        prefix=np.uint64(uid)<<np.uint64(32)
        p=locate(ui_keys,prefix|np.uint64(raw[e]))
        if p>=0 and ui_map[p]>0:
            j=ui_map[p]
            put(UI,j,b,event,ratio[e],organic[e],length[e])
            last_ui[j]=ts[e]
            if first_ui[j]<0: first_ui[j]=ts[e]
            if event<2: last_ratio[j]=ratio[e]
            if event==2: like[j]=1.
            elif event==3: like[j]=0.
            elif event==4: dislike[j]=1.
            elif event==5: dislike[j]=0.
        for a in range(ptr[it],ptr[it+1]):
            q=locate(ua_keys,prefix|np.uint64(artists[a]))
            if q>=0 and ua_map[q]>0:
                j=ua_map[q]
                put(UA,j,b,event,ratio[e],organic[e],length[e])
                last_ua[j]=ts[e]


def save_dense_bins(folder,name,raw,chunk=8192):
    arr=np.lib.format.open_memmap(folder/f'{name}.npy',mode='w+',
                                  dtype=np.float16,shape=raw.shape)
    for start in range(0,len(raw),chunk):
        stop=min(start+chunk,len(raw))
        arr[start:stop]=np.log1p(raw[start:stop]).astype(np.float16)
    arr.flush()
    del arr


def save_sparse_bins(folder,name,raw,chunk=8192):

    nr=len(raw)
    ptr=np.zeros(nr+1,dtype=np.int64)
    for start in range(0,nr,chunk):
        stop=min(start+chunk,nr)
        count=np.count_nonzero(np.any(raw[start:stop]!=0,axis=2),axis=1)
        ptr[start+1:stop+1]=np.cumsum(count,dtype=np.int64)+ptr[start]
    np.save(folder/f'{name}_ptr.npy',ptr)
    nnz=int(ptr[-1])
    bins=np.lib.format.open_memmap(folder/f'{name}_bin.npy',mode='w+',
                     dtype=np.uint8,shape=(nnz,))
    vals=np.lib.format.open_memmap(folder/f'{name}_val.npy',mode='w+',
                     dtype=np.float32,shape=(nnz,N_CHANNELS))
    for start in range(0,nr,chunk):
        stop=min(start+chunk,nr)
        block=raw[start:stop]
        local_row,local_bin=np.nonzero(np.any(block!=0,axis=2))
        off=int(ptr[start])
        sz=len(local_bin)
        bins[off:off+sz]=local_bin.astype(np.uint8)
        vals[off:off+sz]=np.log1p(block[local_row,local_bin])
    bins.flush(); vals.flush()
    del bins,vals
    print(f'  {name}: {nr:,} rows, {nnz:,} nonempty time bins',flush=True)


@njit
def gather_sparse_bins(ptr,bins,vals,rows,nf,nc):
    out=np.zeros((len(rows),nf,nc),np.float32)
    for j in range(len(rows)):
        r=rows[j]
        for k in range(ptr[r],ptr[r+1]):
            b=bins[k]
            for c in range(nc):
                out[j,b,c]=vals[k,c]
    return out


## Исторические задачи

In [5]:
def historical_candidates(cutoff,seed):
    generator=np.random.default_rng(seed)
    hist_end=cutoff-GAP
    n=np.searchsorted(ET,hist_end,side='left')
    hi=np.searchsorted(ET,cutoff+DAY,side='left')
    lo=np.searchsorted(ET,cutoff,side='left')
    counts=np.bincount(EI[:n][EK[:n]==1],minlength=N_ITEMS)
    catalog=np.flatnonzero(counts>0)
    probs=np.power(counts[catalog].astype(np.float64),.75)
    probs/=probs.sum()
    positive=pd.DataFrame({'u':EU[lo:hi][EK[lo:hi]==1],
                           'item':EI_RAW[lo:hi][EK[lo:hi]==1]}).drop_duplicates()
    positive=positive[positive.item.isin(items[catalog])]
    day_positive={int(u):set(g.item.to_numpy().tolist()) for u,g in positive.groupby('u')}
    positive=positive.iloc[generator.permutation(len(positive))]
    positive=positive.groupby('u',sort=False).head(HIST_POS_PER_USER).reset_index(drop=True)
    m=len(positive)

    recent_from=np.searchsorted(ET,hist_end-30*DAY,side='left')
    recent=pd.DataFrame({'u':EU[recent_from:n],'item':EI_RAW[recent_from:n]})
    recent=recent.drop_duplicates(['u','item'],keep='last').groupby('u',sort=False).tail(40)
    recent_pools={int(u):g.item.to_numpy(dtype=np.uint32) for u,g in recent.groupby('u')}
    pop_draws=items[generator.choice(catalog,size=(m,NEG*3),p=probs)]
    result=np.empty((m,NEG+1),np.uint32)
    result[:,0]=positive.item.to_numpy(dtype=np.uint32)
    for j,(u,raw) in enumerate(positive[['u','item']].itertuples(index=False,name=None)):
        forbidden=day_positive[int(u)]
        pool=recent_pools.get(int(u),())
        used={int(raw)}
        cursor=0
        for k in range(1,NEG+1):
            choice=None
            if k%2==0 and len(pool):
                for _ in range(8):
                    q=int(pool[generator.integers(len(pool))])
                    if q not in forbidden and q not in used:
                        choice=q
                        break
            while choice is None and cursor<NEG*3:
                q=int(pop_draws[j,cursor]); cursor+=1
                if q not in forbidden and q not in used: choice=q
            if choice is None:
                forbidden_items=np.fromiter(forbidden|used,dtype=np.uint32)
                available=np.setdiff1d(items[catalog],forbidden_items,assume_unique=False)
                choice=int(generator.choice(available))
            result[j,k]=choice
            used.add(choice)
    return pd.DataFrame({'uid':users[np.repeat(positive.u.to_numpy(dtype=np.int32),NEG+1)],
                         'item_id':result.ravel(),
                         'label':np.tile(np.r_[1,np.zeros(NEG,dtype=np.uint8)],m)})

train_history_end=int(snaps.loc['train'].history_end)
hist_days=pd.DataFrame([dict(name=f'hist_{j}',cutoff=train_history_end-(HIST_DAYS-j)*DAY,
                             history_end=train_history_end-(HIST_DAYS-j)*DAY-GAP)
                        for j in range(HIST_DAYS)])
display(hist_days)
print('Last historical target ends at',hist_days.cutoff.max()+DAY,
      '; main train history ends at',train_history_end)


,name,cutoff,history_end
0,hist_0,24871400,24869600
1,hist_1,24957800,24956000
2,hist_2,25044200,25042400
3,hist_3,25130600,25128800
4,hist_4,25217000,25215200
5,hist_5,25303400,25301600
6,hist_6,25389800,25388000
7,hist_7,25476200,25474400
8,hist_8,25562600,25560800
9,hist_9,25649000,25647200


Last historical target ends at 25735400 ; main train history ends at 25735400


## Снапшоты

In [6]:
STORE_VERSION = 'sparse_fast_verified_v2'
STORE_KEYS = ('u','i','ui_ptr','ui_bin','ui_val','ua_ptr','ua_bin','ua_val',
              'u_last','i_last','ua_last','ui_meta','cand_u','cand_i','cand_ui',
              'offsets','link_ua','common','y')

def candidate_signature(frame):
    fields = ['uid', 'item_id', 'label']
    fields += [key for key in ('retrieval_rank', 'itemknn_score',
                               'decaypop_score', 'retrieval_source') if key in frame]
    values = pd.util.hash_pandas_object(frame[fields], index=False).to_numpy(dtype=np.uint64)
    return hashlib.sha256(values.tobytes()).hexdigest()

def store_metadata(frame, history_end):
    return dict(version=STORE_VERSION, history_end=int(history_end), seed=SEED,
                n_users=N_USERS, n_items=N_ITEMS, n_rows=len(frame),
                candidate_hash=candidate_signature(frame),
                edges_hash=hashlib.sha256(FINE_EDGES.tobytes()).hexdigest(),
                history_bytes=(DATA/'events_sample.parquet').stat().st_size,
                artists_hash=hashlib.sha256(ART.tobytes()).hexdigest())

def store_ready(name, frame, history_end):
    folder = LOCAL / f'store_{name}'
    try:
        if not (folder/'ready.txt').exists():
            return False
        if json.loads((folder/'manifest.json').read_text()) != store_metadata(frame, history_end):
            return False
        if any(not (folder/f'{key}.npy').is_file() for key in STORE_KEYS):
            return False
        y = np.load(folder/'y.npy', mmap_mode='r')
        return len(y) == len(frame) and y.dtype == np.uint8
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return False




def candidate_arrays(frame):
    u = pd.Index(users).get_indexer(frame.uid).astype(np.int32)
    raw = frame.item_id.to_numpy(dtype=np.uint32)
    it = np.searchsorted(items, raw).astype(np.int32)

    pair_keys = (u.astype(np.uint64) << np.uint64(32)) | raw.astype(np.uint64)
    ui_keys, inv_ui = np.unique(pair_keys, return_inverse=True)

    counts = APTR[it + 1] - APTR[it]
    offsets = np.r_[0, np.cumsum(counts, dtype=np.int64)]
    nlinks = int(offsets[-1])

    link_rows = np.repeat(np.arange(len(frame), dtype=np.int32), counts)
    within = np.arange(nlinks, dtype=np.int64) - np.repeat(offsets[:-1], counts)
    art_links = ART[APTR[it[link_rows]] + within]

    ua_per_link = (u[link_rows].astype(np.uint64) << np.uint64(32)) | art_links.astype(np.uint64)
    ua_keys, inv_ua = np.unique(ua_per_link, return_inverse=True)

    if 'retrieval_rank' in frame:
        common = np.column_stack([
            frame.retrieval_rank.to_numpy(dtype=np.float32),
            frame.itemknn_score.to_numpy(dtype=np.float32),
            frame.decaypop_score.to_numpy(dtype=np.float32),
            frame.retrieval_source.eq('itemknn').to_numpy(dtype=np.float32)
        ])
    else:
        common = np.zeros((len(frame), 4), np.float32)

    return dict(
        u=u, it=it,
        y=frame.label.to_numpy(dtype=np.uint8),
        common=common,
        ui_keys=ui_keys,
        inv_ui=inv_ui,
        ua_keys=ua_keys,
        offsets=offsets,
        inv_ua=inv_ua.astype(np.int32)
    )


def load_main_candidates(split):
    c = pd.read_parquet(
        DATA / 'candidates.parquet',
        filters=[('split', '==', split)]
    )
    return c.sort_values(
        ['group_id', 'retrieval_rank'],
        kind='stable'
    ).reset_index(drop=True)


def create_store(name, frame, history_end):

    start = time.perf_counter()
    end = int(history_end)
    n = np.searchsorted(ET, end, side='left')
    x = candidate_arrays(frame)

    available = psutil.virtual_memory().available
    budget = int(min(RAM_MAX_GIB * 1024**3, available * RAM_SHARE))

    bytes_per_row = NF * N_CHANNELS * np.dtype(np.float32).itemsize
    all_rows = N_USERS + N_ITEMS + len(x['ui_keys']) + len(x['ua_keys']) + 2
    direct = all_rows * bytes_per_row < budget * 0.88

    t = time.perf_counter()

    if direct:
        map_ui = np.arange(1, len(x['ui_keys']) + 1, dtype=np.int32)
        map_ua = np.arange(1, len(x['ua_keys']) + 1, dtype=np.int32)
        n_ui, n_ua = len(map_ui), len(map_ua)

        print(f'{name}: direct candidate mapping; no mark_relations pass', flush=True)
    else:
        seen_ui, seen_ua = mark_relations(
            EU[:n], EI_RAW[:n], EI[:n], APTR, ART,
            x['ui_keys'], x['ua_keys']
        )
        map_ui = np.cumsum(seen_ui, dtype=np.int32) * seen_ui
        map_ua = np.cumsum(seen_ua, dtype=np.int32) * seen_ua
        n_ui, n_ua = int(seen_ui.sum()), int(seen_ua.sum())

        del seen_ui, seen_ua
        print(f'{name}: compact observed mapping', flush=True)

    uiref = map_ui[x['inv_ui']]
    link_ua = map_ua[x['inv_ua']]

    print(
        f'{name}: indexing {time.perf_counter()-t:.1f}s, '
        f'events={n:,}, ui={n_ui:,}, ua={n_ua:,}',
        flush=True
    )

    d = LOCAL / f'store_{name}'
    d.mkdir(parents=True, exist_ok=True)

    ram_used = 0
    if name == 'test':
        needed = (N_USERS + N_ITEMS + n_ui + 1) * NF * N_CHANNELS * 4
        if budget < needed:
            raise MemoryError(f'Test requires at least {needed / 2**30:.2f} GiB staging; budget {budget / 2**30:.2f} GiB')

    def tmp(k, shape):
        nonlocal ram_used

        nb = int(np.prod(shape, dtype=np.int64)) * 4

        if ram_used + nb <= budget:
            ram_used += nb
            return np.zeros(shape, dtype=np.float32)

        return np.lib.format.open_memmap(
            d / f'raw_{k}.npy',
            mode='w+',
            dtype=np.float32,
            shape=shape
        )

    U = tmp('u', (N_USERS, NF, N_CHANNELS))
    I = tmp('i', (N_ITEMS, NF, N_CHANNELS))
    UI = tmp('ui', (n_ui + 1, NF, N_CHANNELS))
    UA = tmp('ua', (n_ua + 1, NF, N_CHANNELS))

    print(
        f'{name}: RAM staging {ram_used/2**30:.2f} GiB / '
        f'budget {budget/2**30:.2f} GiB',
        flush=True
    )

    last_u = np.full(N_USERS, -1, np.int64)
    last_i = np.full(N_ITEMS, -1, np.int64)
    last_ui = np.full(n_ui + 1, -1, np.int64)
    first_ui = np.full(n_ui + 1, -1, np.int64)
    last_ratio = np.zeros(n_ui + 1, np.float32)
    like = np.zeros(n_ui + 1, np.float32)
    dislike = np.zeros(n_ui + 1, np.float32)
    last_ua = np.full(n_ua + 1, -1, np.int64)

    t = time.perf_counter()

    aggregate_history_into(
        U, I, UI, UA,
        last_u, last_i, last_ui, first_ui,
        last_ratio, like, dislike, last_ua,
        EU[:n], EI_RAW[:n], EI[:n], ET[:n],
        EK[:n], ER[:n], EO[:n], EL[:n],
        end, FINE_EDGES, APTR, ART,
        x['ui_keys'], x['ua_keys'], map_ui, map_ua
    )

    for a in (U, I, UI, UA):
        if isinstance(a, np.memmap):
            a.flush()

    print(
        f'{name}: event aggregation {time.perf_counter()-t:.1f}s',
        flush=True
    )

    if direct:
        uiref = np.where(last_ui[uiref] >= 0, uiref, 0).astype(np.int32)
        link_ua = np.where(last_ua[link_ua] >= 0, link_ua, 0).astype(np.int32)

    t = time.perf_counter()

    save_dense_bins(d, 'u', U)
    save_dense_bins(d, 'i', I)
    save_sparse_bins(d, 'ui', UI)
    save_sparse_bins(d, 'ua', UA)

    del U, I, UI, UA
    gc.collect()

    for k in ('u', 'i', 'ui', 'ua'):
        p = d / f'raw_{k}.npy'
        if p.exists():
            p.unlink()

    print(
        f'{name}: save and compression {time.perf_counter()-t:.1f}s',
        flush=True
    )

    ui_meta = np.column_stack([
        np.where(last_ui >= 0, np.log1p(np.maximum(0, end-last_ui)/DAY), 0),
        np.where(first_ui >= 0, np.log1p(np.maximum(0, end-first_ui)/DAY), 0),
        last_ratio,
        like,
        dislike,
        (last_ui >= 0).astype(np.float32)
    ]).astype(np.float32)

    arrays = dict(
        u_last=np.where(
            last_u >= 0, np.log1p(np.maximum(0, end-last_u)/DAY), 0
        ).astype(np.float32),
        i_last=np.where(
            last_i >= 0, np.log1p(np.maximum(0, end-last_i)/DAY), 0
        ).astype(np.float32),
        ua_last=np.where(
            last_ua >= 0, np.log1p(np.maximum(0, end-last_ua)/DAY), 0
        ).astype(np.float32),
        ui_meta=ui_meta,
        cand_u=x['u'],
        cand_i=x['it'],
        cand_ui=uiref,
        offsets=x['offsets'],
        link_ua=link_ua,
        y=x['y'],
        common=x['common']
    )

    for k, v in arrays.items():
        np.save(d / f'{k}.npy', v)

    (d / 'manifest.json').write_text(json.dumps(store_metadata(frame, end), indent=2))
    (d / 'ready.txt').write_text(
        f'v3_sparse_fast; history_end={end}; '
        f'count={n}; secs={time.perf_counter()-start:.1f}'
    )

    print(
        name,
        'history events', f'{n:,}',
        'pairs', f'{len(frame):,}',
        'stored ui', f'{n_ui:,}',
        'stored ua', f'{n_ua:,}',
        'seconds', round(time.perf_counter()-start, 1),
        flush=True
    )

    del arrays, x, ui_meta
    gc.collect()



for day in hist_days.itertuples(index=False):
    d = LOCAL / f'store_{day.name}'
    frame = historical_candidates(int(day.cutoff), SEED + int(day.name.split('_')[1]))
    if not store_ready(day.name, frame, int(day.history_end)):
        shutil.rmtree(d, ignore_errors=True)
        create_store(day.name, frame, int(day.history_end))
    del frame
    gc.collect()

for split in ('train', 'valid'):
    d = LOCAL / f'store_{split}'
    frame = load_main_candidates(split)
    if not store_ready(split, frame, int(snaps.loc[split].history_end)):
        shutil.rmtree(d, ignore_errors=True)
        create_store(split, frame, int(snaps.loc[split].history_end))
    del frame
    gc.collect()


hist_0: direct candidate mapping; no mark_relations pass
hist_0: indexing 0.0s, events=44,944,840, ui=321,513, ua=292,877
hist_0: RAM staging 1.39 GiB / budget 3.00 GiB
hist_0: event aggregation 29.4s
  ui: 321,514 rows, 495,168 nonempty time bins
  ua: 292,878 rows, 794,683 nonempty time bins
hist_0: save and compression 7.4s
hist_0 history events 44,944,840 pairs 394,004 stored ui 321,513 stored ua 292,877 seconds 37.1
hist_1: direct candidate mapping; no mark_relations pass
hist_1: indexing 0.0s, events=45,171,606, ui=299,819, ua=273,944
hist_1: RAM staging 1.36 GiB / budget 3.00 GiB
hist_1: event aggregation 25.7s
  ui: 299,820 rows, 469,313 nonempty time bins
  ua: 273,945 rows, 747,991 nonempty time bins
hist_1: save and compression 8.7s
hist_1 history events 45,171,606 pairs 367,614 stored ui 299,819 stored ua 273,944 seconds 34.7
hist_2: direct candidate mapping; no mark_relations pass
hist_2: indexing 0.0s, events=45,380,992, ui=320,861, ua=292,702
hist_2: RAM staging 1.39 GiB

## Энкодер

In [27]:
class RelationStore:
    def __init__(self,name):
        d=LOCAL/f'store_{name}'
        keys=('u','i','ui_ptr','ui_bin','ui_val','ua_ptr','ua_bin','ua_val',
              'u_last','i_last','ua_last','ui_meta',
              'cand_u','cand_i','cand_ui','offsets','link_ua','common','y')
        self.a={k:np.load(d/f'{k}.npy',mmap_mode='r') for k in keys}
        self.n=len(self.a['y'])

    def batch(self,rows):
        a=self.a
        rows=np.asarray(rows,dtype=np.int64)
        uu=a['cand_u'][rows]
        ii=a['cand_i'][rows]
        pp=a['cand_ui'][rows]
        parts=[np.asarray(a['u'][uu],dtype=np.float32),
               np.asarray(a['i'][ii],dtype=np.float32),
               gather_sparse_bins(a['ui_ptr'],a['ui_bin'],a['ui_val'],pp,NF,N_CHANNELS)]
        lengths=a['offsets'][rows+1]-a['offsets'][rows]
        start=a['offsets'][rows]
        within=np.arange(int(lengths.sum()))-np.repeat(np.cumsum(lengths)-lengths,lengths)
        indices=a['link_ua'][np.repeat(start,lengths)+within]
        ua_sum=np.zeros((len(rows),NF,N_CHANNELS),np.float32)
        ua_max=np.zeros_like(ua_sum)
        artist_age=np.zeros(len(rows),np.float32)
        n_known=np.zeros(len(rows),np.float32)
        if len(indices):
            vals=np.expm1(gather_sparse_bins(a['ua_ptr'],a['ua_bin'],a['ua_val'],
                                        indices,NF,N_CHANNELS))
            nonempty=np.flatnonzero(lengths>0)
            starts=np.r_[0,np.cumsum(lengths)][nonempty]
            ua_sum[nonempty]=np.log1p(np.add.reduceat(vals,starts,axis=0))
            ua_max[nonempty]=np.log1p(np.maximum.reduceat(vals,starts,axis=0))
            known=(indices>0)
            ages=np.where(known,np.asarray(a['ua_last'][indices]),np.inf)
            artist_age[nonempty]=np.minimum.reduceat(ages,starts)
            n_known[nonempty]=np.add.reduceat(known.astype(np.float32),starts)
            artist_age[~np.isfinite(artist_age)]=0
        parts.extend([ua_sum,ua_max])
        side=np.column_stack([a['ui_meta'][pp],a['u_last'][uu],a['i_last'][ii],
            np.log1p(lengths.astype(np.float32)),artist_age,np.log1p(n_known)]).astype(np.float32)
        return np.stack(parts,axis=1),side


def torch_batch(store,rows):
    history,side=store.batch(rows)
    return (torch.from_numpy(history).to(DEVICE),
            torch.from_numpy(side).to(DEVICE))

class AdaptiveBinsEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.boundary_logits=nn.Parameter(torch.zeros(5,N_COARSE))
        self.register_buffer('fine_centers',torch.tensor(FINE_CENTERS))
        self.temperature=.22
        width=N_COARSE*N_CHANNELS+N_CHANNELS
        self.parts=nn.ModuleList([nn.Sequential(nn.Linear(width,48),nn.GELU(),
                            nn.Linear(48,48),nn.GELU()) for _ in range(5)])
        self.fuse=nn.Sequential(nn.Linear(48*8+11,160),nn.GELU(),
                   nn.LayerNorm(160),nn.Linear(160,Z_DIM),nn.GELU())
        self.head=nn.Sequential(nn.Linear(Z_DIM,64),nn.GELU(),nn.Linear(64,1))

    def soft_bins(self):

        widths=.95*torch.softmax(self.boundary_logits,dim=-1)+.05/N_COARSE
        x0,x1=0.,float(np.log1p(365))
        bounds=x0+(x1-x0)*torch.cumsum(widths,dim=-1)[:,:-1]
        cdf=torch.sigmoid((self.fine_centers[None,:,None]-bounds[:,None,:])/self.temperature)
        membership=torch.cat([1-cdf[:,:,:1],cdf[:,:,:-1]-cdf[:,:,1:],cdf[:,:,-1:]],dim=-1)
        return membership,bounds

    def forward(self,mem,side):

        hist=torch.expm1(mem.clamp(min=0))
        weights,_=self.soft_bins()
        pooled=torch.einsum('brfc,rfk->brkc',hist,weights)
        absolute=torch.log1p(hist.sum(dim=2))
        dense=torch.cat([torch.log1p(pooled).flatten(2),absolute],dim=-1)
        p=[layer(dense[:,j]) for j,layer in enumerate(self.parts)]
        u,it,ui,ua,ua_max=p
        z=self.fuse(torch.cat([*p,u*it,u*ua,ui*ua,side],dim=-1))
        return self.head(z).squeeze(-1),z

train_store=RelationStore('train')
valid_store=RelationStore('valid')
model=AdaptiveBinsEncoder().to(DEVICE)
print('Encoder parameters:',sum(p.numel() for p in model.parameters()))
print('Initial learned boundaries in days:')
with torch.no_grad():
    print(np.expm1(model.soft_bins()[1].cpu().numpy()).round(2))


Encoder parameters: 107039
Initial learned boundaries in days:
[[  1.67   6.15  18.13  50.17 135.85]
 [  1.67   6.15  18.13  50.17 135.85]
 [  1.67   6.15  18.13  50.17 135.85]
 [  1.67   6.15  18.13  50.17 135.85]
 [  1.67   6.15  18.13  50.17 135.85]]


## Обучение

In [28]:
def historical_ndcg(scores,n_neg):
    s=scores.reshape(-1,n_neg+1)
    rank=(s[:,1:]>s[:,:1]).sum(axis=1)
    k=min(10,n_neg+1)
    return np.where(rank<k,1/np.log2(rank+2),0.).mean()

@torch.no_grad()
def predict_net(net,store,rows,batch_size=INFER_BATCH,features=False):
    net.eval()
    arr=np.empty((len(rows),Z_DIM+1) if features else len(rows),np.float32)
    for left in range(0,len(rows),batch_size):
        rr=np.asarray(rows[left:left+batch_size],dtype=np.int64)
        score,z=net(*torch_batch(store,rr))
        n=len(rr)
        if features:
            arr[left:left+n,:Z_DIM]=z.cpu().numpy()
            arr[left:left+n,-1]=score.cpu().numpy()
        else:
            arr[left:left+n]=score.cpu().numpy()
    return arr

def historical_eval(net,store,max_groups=4000):
    groups=min(store.n//(NEG+1),max_groups)
    group_ids=np.linspace(0,store.n//(NEG+1)-1,groups,dtype=np.int64)
    rr=(group_ids[:,None]*(NEG+1)+np.arange(NEG+1)).ravel()
    s=predict_net(net,store,rr)
    return historical_ndcg(s,NEG)

hist_train_names=hist_days.name.iloc[:-1].tolist()
hist_dev=RelationStore(hist_days.name.iloc[-1])
encoder=AdaptiveBinsEncoder().to(DEVICE)
opt=torch.optim.AdamW(encoder.parameters(),lr=ENC_LR,weight_decay=1e-3)
logs=[]
best=-1.
bad_epochs=0
for epoch in range(ENC_EPOCHS):
    encoder.train()
    losses=[]
    shuffled=rng.permutation(hist_train_names)
    for name in shuffled:
        st=RelationStore(name)
        n_groups=st.n//(NEG+1)
        order=rng.permutation(n_groups)
        for left in tqdm(range(0,n_groups,BATCH_GROUPS),desc=f'epoch {epoch+1}: {name}',leave=False):
            batch_ids=order[left:left+BATCH_GROUPS]
            rows=(batch_ids[:,None]*(NEG+1)+np.arange(NEG+1)).ravel()
            scores,_=encoder(*torch_batch(st,rows))
            loss=F.cross_entropy(scores.reshape(len(batch_ids),NEG+1),
                     torch.zeros(len(batch_ids),device=DEVICE,dtype=torch.long))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(encoder.parameters(),1.)
            opt.step()
            losses.append(float(loss.item()))
        del st
    score=historical_eval(encoder,hist_dev)
    entry=dict(epoch=epoch+1,train_loss=float(np.mean(losses)),hist_dev_ndcg10=float(score))
    logs.append(entry)
    print('Epoch',epoch+1,'loss',round(entry['train_loss'],4),
          'historical dev NDCG@10',round(score,5),flush=True)
    if score>best+1e-5:
        best=score
        bad_epochs=0
        torch.save(encoder.state_dict(),OUT/'encoder.pt')
    else:
        bad_epochs+=1
        if bad_epochs>=ENC_PATIENCE:
            print('Historical early stopping')
            break
pd.DataFrame(logs).to_csv(OUT/'encoder_training.csv',index=False)
encoder.load_state_dict(torch.load(OUT/'encoder.pt',map_location=DEVICE,weights_only=True))
encoder.eval()
with torch.no_grad():
    boundaries=np.expm1(encoder.soft_bins()[1].cpu().numpy())
pd.DataFrame(boundaries,index=['user','item','user_item','user_artist_sum','user_artist_max'],
    columns=[f'cut_{j+1}_days' for j in range(N_COARSE-1)]).to_csv(OUT/'learned_boundaries.csv')
print('Learned boundaries (days):')
display(pd.read_csv(OUT/'learned_boundaries.csv'))


epoch 1: hist_3:   0%|          | 0/80 [00:00<?, ?it/s]

epoch 1: hist_0:   0%|          | 0/79 [00:00<?, ?it/s]

epoch 1: hist_7:   0%|          | 0/75 [00:00<?, ?it/s]

epoch 1: hist_2:   0%|          | 0/79 [00:00<?, ?it/s]

epoch 1: hist_4:   0%|          | 0/81 [00:00<?, ?it/s]

...

epoch 16: hist_0:   0%|          | 0/79 [00:00<?, ?it/s]

epoch 16: hist_6:   0%|          | 0/76 [00:00<?, ?it/s]

epoch 16: hist_2:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 16 loss 0.9372 historical dev NDCG@10 0.83993
Historical early stopping
Learned boundaries (days):


,Unnamed: 0,cut_1_days,cut_2_days,cut_3_days,cut_4_days,cut_5_days
0,user,1.099249,3.582718,11.229781,36.699060,120.501870
1,item,1.635267,6.579147,17.444391,46.460210,125.060196
2,user_item,1.378970,4.840265,14.960426,43.024284,123.473680
3,user_artist_sum,1.316656,5.335434,16.372677,45.149190,130.136950
4,user_artist_max,1.645998,6.695809,18.047964,53.278385,148.670930


## Признаки

In [29]:
def checkpoint_hash(file):
    checksum = hashlib.sha256()
    with open(file, 'rb') as source:
        for data in iter(lambda: source.read(1 << 20), b''):
            checksum.update(data)
    return checksum.hexdigest()

def feature_token(file, shape, checkpoint, split):
    return dict(shape=list(shape), dtype='float16',
                checkpoint=checkpoint_hash(checkpoint),
                history_end=int(snaps.loc[split].history_end),
                file_bytes=file.stat().st_size)

def feature_ready(file, shape, checkpoint, split):
    marker = file.with_suffix(file.suffix + '.done.json')
    try:
        if not marker.exists() or not file.exists():
            return False
        meta = json.loads(marker.read_text())
        arr = np.load(file, mmap_mode='r')
        return (arr.shape == tuple(shape) and arr.dtype == np.float16
                and meta == feature_token(file, shape, checkpoint, split))
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return False

def finish_feature(tmp, filename, shape, checkpoint, split):
    os.replace(tmp, filename)
    marker = filename.with_suffix(filename.suffix + '.done.json')
    staged = marker.with_suffix('.partial.json')
    staged.write_text(json.dumps(feature_token(filename, shape, checkpoint, split), indent=2))
    os.replace(staged, marker)

def extract_features(name, store):
    filename = LOCAL / f'learned_{name}.npy'
    tmp = LOCAL / f'learned_{name}.partial.npy'
    shape = (store.n, Z_DIM + 1)
    result = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.float16, shape=shape)
    for left in tqdm(range(0, store.n, INFER_BATCH), desc=f'Extract {name}'):
        right = min(store.n, left + INFER_BATCH)
        result[left:right] = predict_net(encoder, store, np.arange(left, right), features=True)
    result.flush()
    del result
    finish_feature(tmp, filename, shape, OUT/'encoder.pt', name)
    print(name, 'features:', store.n, 'rows')

extract_features('train', train_store)
extract_features('valid', valid_store)
shutil.copy2(LOCAL/'learned_train.npy', OUT/'learned_train.npy')
shutil.copy2(LOCAL/'learned_valid.npy', OUT/'learned_valid.npy')


Extract train:   0%|          | 0/1465 [00:00<?, ?it/s]

train features: 6000000 rows


Extract valid:   0%|          | 0/1465 [00:00<?, ?it/s]

valid features: 6000000 rows


PosixPath('/content/drive/MyDrive/TLab_Yambda/adaptive_bins_autofe_sparse_v3/learned_valid.npy')

## CatBoost

In [30]:
cb_y=np.asarray(train_store.a['y']).reshape(N_USERS,C)
negative=[]
neg_rng=np.random.default_rng(SEED)
for u in range(N_USERS):
    choices=np.flatnonzero(cb_y[u]==0)
    negative.append(u*C+neg_rng.choice(choices,size=min(CB_NEG_PER_USER,len(choices)),replace=False))
cb_rows=np.sort(np.r_[np.flatnonzero(cb_y.ravel()),np.concatenate(negative)]).astype(np.int64)
valid_users=np.sort(np.random.default_rng(SEED).choice(N_USERS,
                            size=min(CB_EVAL_USERS,N_USERS),replace=False))
valid_rows=(valid_users[:,None]*C+np.arange(C)).ravel()
print('Matched train rows:',len(cb_rows),'Validation early stopping rows:',len(valid_rows))

feature_sets=json.loads((R2/'feature_sets.json').read_text())
manual_cols=feature_sets['R2']
raw_common=['retrieval_rank','itemknn_score','decaypop_score','source_is_itemknn']
learned_cols=[f'latent_{j:02d}' for j in range(Z_DIM)]+['neural_score']

def load_manual(split,rows):
    f=R2/'features'/f'features_{split}.parquet'
    wanted=list(dict.fromkeys(manual_cols+['label']))
    rows=np.asarray(rows,dtype=np.int64)
    pf=pq.ParquetFile(f)
    chunks=[]
    offset=0
    for b in pf.iter_batches(batch_size=100_000,columns=wanted):
        stop=offset+b.num_rows
        lo=np.searchsorted(rows,offset)
        hi=np.searchsorted(rows,stop)
        if lo<hi:
            chunks.append(b.to_pandas().iloc[rows[lo:hi]-offset])
        offset=stop
    return pd.concat(chunks,ignore_index=True)

man_train=load_manual('train',cb_rows)
man_eval=load_manual('valid',valid_rows)
mem_train=np.load(LOCAL/'learned_train.npy',mmap_mode='r')
mem_valid=np.load(LOCAL/'learned_valid.npy',mmap_mode='r')
manual_tr=man_train[manual_cols].to_numpy(dtype=np.float32)
manual_va=man_eval[manual_cols].to_numpy(dtype=np.float32)
learned_tr=np.column_stack([mem_train[cb_rows].astype(np.float32),
                  np.asarray(train_store.a['common'][cb_rows],dtype=np.float32)])
learned_va=np.column_stack([mem_valid[valid_rows].astype(np.float32),
                  np.asarray(valid_store.a['common'][valid_rows],dtype=np.float32)])
y_train=np.asarray(train_store.a['y'][cb_rows],dtype=np.int8)
y_valid=np.asarray(valid_store.a['y'][valid_rows],dtype=np.int8)

def fit_catboost(name,X_train,X_valid):
    model=CatBoostClassifier(iterations=CB_TREES,depth=CB_DEPTH,learning_rate=CB_LR,
        loss_function='Logloss',eval_metric='Logloss',random_seed=SEED,
        task_type='GPU' if DEVICE=='cuda' else 'CPU',verbose=200)
    print(name,X_train.shape,X_valid.shape)
    model.fit(X_train,y_train,eval_set=(X_valid,y_valid),early_stopping_rounds=120)
    model.save_model(str(OUT/f'{name}.cbm'))
    return model

manual=fit_catboost('R2_matched',manual_tr,manual_va)
automatic=fit_catboost('A1_adaptive',learned_tr,learned_va)
del manual_tr,manual_va,learned_tr,learned_va,man_train,man_eval
gc.collect()

def assert_alignment(split, store):
    cols = ['uid', 'item_id', 'group_id', 'retrieval_rank']
    stream = pq.ParquetFile(R2/'features'/f'features_{split}.parquet')
    offset = 0
    for batch in stream.iter_batches(batch_size=100_000, columns=cols):
        d = batch.to_pandas()
        r = slice(offset, offset + len(d))
        u = np.asarray(store.a['cand_u'][r], dtype=np.intp)
        i = np.asarray(store.a['cand_i'][r], dtype=np.intp)
        rank = np.asarray(store.a['common'][r, 0], dtype=np.int32)
        group = int(snaps.loc[split].snapshot_id) * N_USERS + u
        if not (np.array_equal(d.uid.to_numpy(), users[u])
                and np.array_equal(d.item_id.to_numpy(), items[i])
                and np.array_equal(d.group_id.to_numpy(), group)
                and np.array_equal(d.retrieval_rank.to_numpy(), rank)):
            raise ValueError(f'Candidate order mismatch: {split}, row {offset}')
        offset += len(d)
    assert offset == store.n, f'Candidate count mismatch: {split}'

assert_alignment('train', train_store)
assert_alignment('valid', valid_store)


Matched train rows: 1036912 Validation early stopping rows: 307200
R2_matched (1036912, 65) (307200, 65)
0:	learn: 0.5974327	test: 0.5924503	best: 0.5924503 (0)	total: 72ms	remaining: 35.9s
200:	learn: 0.0918739	test: 0.0417417	best: 0.0417417 (200)	total: 12s	remaining: 17.8s
400:	learn: 0.0862927	test: 0.0408453	best: 0.0408453 (400)	total: 23.8s	remaining: 5.88s
499:	learn: 0.0841925	test: 0.0405184	best: 0.0405158 (496)	total: 29.4s	remaining: 0us
bestTest = 0.04051583608
bestIteration = 496
Shrink model to first 497 iterations.
A1_adaptive (1036912, 69) (307200, 69)
0:	learn: 0.5942731	test: 0.5896981	best: 0.5896981 (0)	total: 40.4ms	remaining: 20.2s
200:	learn: 0.0940592	test: 0.0418744	best: 0.0418744 (200)	total: 4.67s	remaining: 6.94s
400:	learn: 0.0900578	test: 0.0415538	best: 0.0415530 (397)	total: 9.12s	remaining: 2.25s
499:	learn: 0.0884865	test: 0.0414419	best: 0.0414419 (499)	total: 11.3s	remaining: 0us
bestTest = 0.04144193331
bestIteration = 499


1962

## Подбор

In [48]:
def future_totals(split):
    s=snaps.loc[split]
    mask=(ET>=int(s.cutoff))&(ET<int(s.end))&(EK==1)
    frame=pd.DataFrame({'u':EU[mask],'item':EI_RAW[mask]}).drop_duplicates()
    return np.bincount(frame.u.to_numpy(dtype=np.int32),minlength=N_USERS)

from sklearn.model_selection import ParameterSampler

params = {
    'depth': [4, 5, 6, 7, 8],
    'learning_rate': [0.03, 0.05, 0.08],
    'l2_leaf_reg': [5, 10, 30]
}
truth_valid = future_totals('valid')

def tune(name, X_train, X_valid, n_trials=10):
    labels = y_valid.reshape(-1, C)
    total = truth_valid[valid_users]
    discount = 1 / np.log2(np.arange(2, 22))
    ideal = np.r_[0, discount.cumsum()]
    results = []

    for p in tqdm(list(ParameterSampler(params, n_iter=n_trials, random_state=SEED)), desc=name):
        model = CatBoostClassifier(
            iterations=2500, loss_function='Logloss',task_type='GPU' if DEVICE=='cuda' else 'CPU',
            random_seed=SEED, verbose=400, **p
        )
        model.fit(X_train, y_train, eval_set=(X_valid, y_valid),
                  early_stopping_rounds=150)

        scores = model.predict_proba(X_valid)[:, 1].reshape(-1, C)
        top = np.argsort(-scores, axis=1)[:, :20]
        dcg = (np.take_along_axis(labels, top, axis=1) * discount).sum(axis=1)
        ndcg = np.mean(np.where(total > 0, dcg / np.maximum(ideal[np.minimum(total, 20)], 1), 0))

        results.append((ndcg, p, model.tree_count_))
        if ndcg == max(r[0] for r in results):
            model.save_model(str(OUT / f'{name}_best.cbm'))

    result = pd.DataFrame(results, columns=['ndcg@20', 'params', 'trees'])
    result.to_csv(OUT / f'{name}_search.csv', index=False)
    return result.sort_values('ndcg@20', ascending=False)


In [24]:
r2_train = load_manual('train', cb_rows)[manual_cols]
r2_valid = load_manual('valid', valid_rows)[manual_cols]

r2_search = tune('R2', r2_train, r2_valid, n_trials=15)
display(r2_search)


R2:   0%|          | 0/15 [00:00<?, ?it/s]

0:	learn: 0.6342988	test: 0.6312651	best: 0.6312651 (0)	total: 41.3ms	remaining: 1m 43s
400:	learn: 0.0909415	test: 0.0413838	best: 0.0413838 (400)	total: 8.77s	remaining: 45.9s
800:	learn: 0.0851968	test: 0.0403444	best: 0.0403428 (799)	total: 17.5s	remaining: 37.1s
1200:	learn: 0.0811203	test: 0.0397223	best: 0.0397216 (1197)	total: 26.1s	remaining: 28.3s
1600:	learn: 0.0779110	test: 0.0392890	best: 0.0392890 (1600)	total: 34.8s	remaining: 19.5s
2000:	learn: 0.0752301	test: 0.0389808	best: 0.0389804 (1995)	total: 43.5s	remaining: 10.9s
2400:	learn: 0.0728266	test: 0.0386981	best: 0.0386959 (2399)	total: 52.4s	remaining: 2.16s
2499:	learn: 0.0722583	test: 0.0386132	best: 0.0386079 (2489)	total: 54.6s	remaining: 0us
bestTest = 0.0386078612
bestIteration = 2489
Shrink model to first 2490 iterations.
0:	learn: 0.6002357	test: 0.5953030	best: 0.5953030 (0)	total: 26.4ms	remaining: 1m 6s
400:	learn: 0.0928962	test: 0.0416681	best: 0.0416681 (400)	total: 6.58s	remaining: 34.4s
800:	learn: 0

,ndcg@20,params,trees
11,0.096872,"{'learning_rate': 0.03, 'l2_leaf_reg': 30, 'de...",2497
1,0.096341,"{'learning_rate': 0.05, 'l2_leaf_reg': 30, 'de...",2486
14,0.096276,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",1202
0,0.095880,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2490
13,0.095865,"{'learning_rate': 0.05, 'l2_leaf_reg': 5, 'dep...",1966
4,0.095301,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",2500
3,0.094906,"{'learning_rate': 0.05, 'l2_leaf_reg': 30, 'de...",2497
2,0.094840,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",1603
7,0.094475,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2497
12,0.093671,"{'learning_rate': 0.08, 'l2_leaf_reg': 10, 'de...",2500


In [31]:
a1_train = np.column_stack([np.load(LOCAL / 'learned_train.npy', mmap_mode='r')[cb_rows],
    train_store.a['common'][cb_rows]
])
a1_valid = np.column_stack([np.load(LOCAL / 'learned_valid.npy', mmap_mode='r')[valid_rows],
    valid_store.a['common'][valid_rows]
])

a1_search = tune('A1', a1_train, a1_valid, n_trials=15)
display(a1_search)


A1:   0%|          | 0/15 [00:00<?, ?it/s]

0:	learn: 0.6322959	test: 0.6295272	best: 0.6295272 (0)	total: 40.1ms	remaining: 1m 40s
400:	learn: 0.0931596	test: 0.0416992	best: 0.0416987 (399)	total: 9.15s	remaining: 47.9s
800:	learn: 0.0891018	test: 0.0413878	best: 0.0413878 (800)	total: 18.2s	remaining: 38.5s
1200:	learn: 0.0859907	test: 0.0411908	best: 0.0411901 (1198)	total: 27.2s	remaining: 29.4s
1600:	learn: 0.0833899	test: 0.0410090	best: 0.0410090 (1600)	total: 36.3s	remaining: 20.4s
2000:	learn: 0.0811116	test: 0.0408756	best: 0.0408748 (1993)	total: 45.4s	remaining: 11.3s
2400:	learn: 0.0789838	test: 0.0407610	best: 0.0407597 (2398)	total: 54.6s	remaining: 2.25s
2499:	learn: 0.0784750	test: 0.0407434	best: 0.0407434 (2499)	total: 56.9s	remaining: 0us
bestTest = 0.04074337324
bestIteration = 2499
0:	learn: 0.5961409	test: 0.5917901	best: 0.5917901 (0)	total: 34ms	remaining: 1m 25s
400:	learn: 0.0948568	test: 0.0420887	best: 0.0420876 (398)	total: 6.96s	remaining: 36.4s
800:	learn: 0.0919170	test: 0.0417637	best: 0.041763

,ndcg@20,params,trees
1,0.096034,"{'learning_rate': 0.05, 'l2_leaf_reg': 30, 'de...",2497
11,0.095424,"{'learning_rate': 0.03, 'l2_leaf_reg': 30, 'de...",2498
9,0.094986,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2496
10,0.094900,"{'learning_rate': 0.03, 'l2_leaf_reg': 30, 'de...",2475
14,0.094353,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",1326
6,0.094031,"{'learning_rate': 0.05, 'l2_leaf_reg': 10, 'de...",2500
7,0.093850,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2491
13,0.093824,"{'learning_rate': 0.05, 'l2_leaf_reg': 5, 'dep...",2421
8,0.093764,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",2025
0,0.092440,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2500


## Валидация

In [33]:
def ranking_report(scores,store,total,name):
    score=np.asarray(scores,dtype=np.float32).reshape(N_USERS,C)
    labels=np.asarray(store.a['y']).reshape(N_USERS,C)
    ranked=np.argsort(-score,axis=1,kind='stable')[:,:100]
    found=np.take_along_axis(labels,ranked,axis=1)
    discount=1/np.log2(np.arange(2,102))
    dcg=np.cumsum(found*discount,axis=1)
    ideal=np.r_[0,np.cumsum(discount)]
    candidate_found=labels.sum(axis=1)
    out=dict(model=name,
        roc_auc=roc_auc_score(labels.ravel(),score.ravel()),
        logloss=log_loss(labels.ravel(),np.clip(score.ravel(),1e-7,1-1e-7)) if name!='R0' else np.nan)
    for k in (5,10,20,100):
        value=dcg[:,k-1]
        out[f'ndcg@{k}']=np.where(total>0,value/np.maximum(ideal[np.minimum(total,k)],1),0).mean()
        out[f'ndcg@{k}_conditional']=np.where(candidate_found>0,
              value/np.maximum(ideal[np.minimum(candidate_found,k)],1),0)[candidate_found>0].mean()
        out[f'recall@{k}']=np.where(total>0,found[:,:k].sum(axis=1)/np.maximum(total,1),0).mean()
    return out

def predict_manual_full(split,model):
    filename=R2/'features'/f'features_{split}.parquet'
    values=[]
    for b in tqdm(pq.ParquetFile(filename).iter_batches(batch_size=100_000,columns=manual_cols),
                  desc=f'Manual {split}'):
        frame=b.to_pandas()
        values.append(model.predict_proba(frame[manual_cols])[:,1].astype(np.float32))
    return np.concatenate(values)

def predict_adaptive_full(split,model):
    st=RelationStore(split)
    feat=np.load(LOCAL/f'learned_{split}.npy',mmap_mode='r')
    result=np.empty(st.n,np.float32)
    for left in tqdm(range(0,st.n,100_000),desc=f'AutoFE {split}'):
        right=min(st.n,left+100_000)
        x=np.column_stack([feat[left:right].astype(np.float32),
                           np.asarray(st.a['common'][left:right],dtype=np.float32)])
        result[left:right]=model.predict_proba(x)[:,1].astype(np.float32)
    return result

r2_best = CatBoostClassifier()
r2_best.load_model(str(OUT / 'R2_best.cbm'))

a1_best = CatBoostClassifier()
a1_best.load_model(str(OUT / 'A1_best.cbm'))

truth_valid = future_totals('valid')

s2 = predict_manual_full('valid', r2_best)
s1 = predict_adaptive_full('valid', a1_best)
scores = {'R2': s2, 'AutoFE': s1}

results = pd.DataFrame([
    ranking_report(s2, valid_store, truth_valid, 'R2_tuned'),
    ranking_report(s1, valid_store, truth_valid, 'A1_tuned')
])

display(results)
results.to_csv(OUT/'validation_metrics.csv',index=False)
np.save(OUT/'scores_R2_valid.npy',s2)
np.save(OUT/'scores_A1_valid.npy',s1)


Manual valid: 0it [00:00, ?it/s]

AutoFE valid:   0%|          | 0/60 [00:00<?, ?it/s]

,model,roc_auc,logloss,ndcg@5,ndcg@5_conditional,recall@5,ndcg@10,ndcg@10_conditional,recall@10,ndcg@20,ndcg@20_conditional,recall@20,ndcg@100,ndcg@100_conditional,recall@100
0,R2_tuned,0.914900,0.040061,0.103298,0.286012,0.031639,0.094611,0.296224,0.046771,0.088879,0.323234,0.064349,0.095825,0.423476,0.112844
1,A1_tuned,0.915957,0.041338,0.102687,0.284657,0.032182,0.093553,0.294472,0.047111,0.087792,0.321668,0.064181,0.095656,0.424497,0.114092


## История кандидатов

In [41]:
SEQ_L, SEQ_EPOCHS = 50, 10
names = hist_days.name.tolist() + ['train', 'valid', 'test']
ends = np.r_[hist_days.history_end, [snaps.loc[s].history_end for s in ('train', 'valid', 'test')]].astype(np.int64)
order = np.argsort(ends)

@njit
def last_tracks(u, item, ts, event, cutoffs, n_users, L):
    buf = np.zeros((n_users, L), np.int32)
    tbuf = np.zeros((n_users, L), np.int64)
    count = np.zeros(n_users, np.int32)
    ids = np.zeros((len(cutoffs), n_users, L), np.int32)
    ages = np.zeros_like(ids, dtype=np.float32)
    p = 0
    for k, cutoff in enumerate(cutoffs):
        while p < len(ts) and ts[p] < cutoff:
            if event[p] == 1:
                a = u[p]; j = count[a] % L
                buf[a, j], tbuf[a, j] = item[p] + 1, ts[p]
                count[a] += 1
            p += 1
        for a in range(n_users):
            for j in range(min(count[a], L)):
                q = (count[a] - j - 1) % L
                ids[k, a, j] = buf[a, q]
                ages[k, a, j] = (cutoff - tbuf[a, q]) / DAY
    return ids, ages

ids, ages = last_tracks(EU, EI, ET, EK, ends[order], N_USERS, SEQ_L)
hist_id = {names[i]: ids[j] for j, i in enumerate(order)}
hist_age = {names[i]: ages[j] for j, i in enumerate(order)}
del ids, ages


class CandidateSeq(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(N_ITEMS + 1, 32, padding_idx=0, sparse=True)
        self.time = nn.Linear(1, 1)
        self.proj = nn.Linear(96, 5)
        self.head = nn.Linear(5, 1)

    def forward(self, item, history, age):
        q, h = self.emb(item + 1), self.emb(history)
        mask = (history > 0) & (history != item[:, None] + 1)
        logits = (q[:, None] * h).sum(-1) / 32**0.5
        logits += self.time(torch.log1p(age)[..., None]).squeeze(-1)
        w = logits.masked_fill(~mask, -1e4).softmax(1) * mask
        w = w / w.sum(1, keepdim=True).clamp_min(1e-8)
        z = (w[..., None] * h).sum(1)
        features = torch.tanh(self.proj(torch.cat([q, z, q*z], dim=1)))
        return self.head(features).squeeze(-1), features

def seq_forward(net, st, name, rows):
    u = st.a['cand_u'][rows]
    return net(torch.as_tensor(st.a['cand_i'][rows], device=DEVICE, dtype=torch.long),
               torch.as_tensor(hist_id[name][u], device=DEVICE, dtype=torch.long),
               torch.as_tensor(hist_age[name][u], device=DEVICE))


## Обучение Seq

In [42]:
seq = CandidateSeq().to(DEVICE)
opt_emb = torch.optim.SparseAdam(seq.emb.parameters(), lr=.001)
opt_head = torch.optim.AdamW(
    list(seq.time.parameters()) + list(seq.proj.parameters()) +
    list(seq.head.parameters()), lr=.003
)
dev_name = hist_days.name.iloc[-1]
dev = RelationStore(dev_name)
best, bad = -1., 0

@torch.no_grad()
def seq_predict(name, st, rows, features=False):
    seq.eval()
    result = []
    for p in range(0, len(rows), 4096):
        score, feat = seq_forward(seq, st, name, rows[p:p+4096])
        result.append((torch.cat([feat, score[:, None]], 1) if features else score).cpu().numpy())
    return np.concatenate(result)

for epoch in range(SEQ_EPOCHS):
    seq.train(); losses = []
    for name in rng.permutation(hist_days.name.iloc[-6:-1]):
        st = RelationStore(name)
        groups = rng.permutation(st.n // (NEG + 1))
        for p in tqdm(range(0, len(groups), 256), desc=f'Seq {epoch+1}: {name}', leave=False):
            g = groups[p:p+256]
            rows = (g[:, None]*(NEG+1) + np.arange(NEG+1)).ravel()
            score, _ = seq_forward(seq, st, name, rows)
            loss = F.cross_entropy(score.reshape(len(g), NEG+1),
                                   torch.zeros(len(g), device=DEVICE, dtype=torch.long))
            opt_emb.zero_grad(); opt_head.zero_grad()
            loss.backward(); opt_emb.step(); opt_head.step()
            losses.append(loss.item())
    g = min(4000, dev.n // (NEG+1))
    rows = (np.arange(g)[:, None]*(NEG+1) + np.arange(NEG+1)).ravel()
    score = seq_predict(dev_name, dev, rows).reshape(g, NEG+1)
    rank = (score[:, 1:] >= score[:, :1]).sum(1)
    ndcg = np.where(rank < 10, 1/np.log2(rank+2), 0).mean()
    print(f'Seq epoch {epoch+1}: loss={np.mean(losses):.4f}, dev NDCG@10={ndcg:.5f}')
    if ndcg > best + 1e-4:
        best, bad = ndcg, 0
        torch.save(seq.state_dict(), OUT/'seq_encoder.pt')
    else:
        bad += 1
        if bad == 2: break

seq.load_state_dict(torch.load(OUT/'seq_encoder.pt', map_location=DEVICE, weights_only=True))
for name in ('train', 'valid'):
    st = RelationStore(name)
    dest = LOCAL/f'seq_{name}.npy'
    tmp = LOCAL/f'seq_{name}.partial.npy'
    out = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.float16, shape=(st.n, 6))
    for p in tqdm(range(0, st.n, 4096), desc=f'Seq features {name}'):
        rows = np.arange(p, min(p+4096, st.n))
        out[rows] = seq_predict(name, st, rows, features=True).astype(np.float16)
    out.flush(); del out
    finish_feature(tmp, dest, (st.n, 6), OUT/'seq_encoder.pt', name)
    shutil.copy2(dest, OUT/f'seq_{name}.npy')


Seq 1: hist_5:   0%|          | 0/115 [00:00<?, ?it/s]

Seq 1: hist_6:   0%|          | 0/114 [00:00<?, ?it/s]

Seq 1: hist_8:   0%|          | 0/108 [00:00<?, ?it/s]

Seq 1: hist_7:   0%|          | 0/112 [00:00<?, ?it/s]

Seq 1: hist_4:   0%|          | 0/122 [00:00<?, ?it/s]

...

Seq 6: hist_7:   0%|          | 0/112 [00:00<?, ?it/s]

Seq 6: hist_5:   0%|          | 0/115 [00:00<?, ?it/s]

Seq epoch 6: loss=2.2260, dev NDCG@10=0.36554


Seq features train:   0%|          | 0/1465 [00:00<?, ?it/s]

Seq features valid:   0%|          | 0/1465 [00:00<?, ?it/s]

## Ручной контроль

In [50]:
def manual_seq(split, rows):
    st = globals()[f'{split}_store']
    u, i = st.a['cand_u'][rows], st.a['cand_i'][rows]
    h, age = hist_id[split][u], hist_age[split][u]
    m = h == i[:, None] + 1
    last = np.where(m, age, np.inf).min(axis=1)

    return np.column_stack([
        m.sum(1), m[:, :5].sum(1),
        (m & (age <= 7)).sum(1),
        (m & (age <= 30)).sum(1),
        (m * np.exp(-age / 7)).sum(1),
        (m * np.exp(-age / 60)).sum(1),
        np.log1p(np.where(np.isfinite(last), last, 1000)),
        m[:, 0]
    ]).astype(np.float32)

Xtr = np.column_stack([
    load_manual('train', cb_rows)[manual_cols],
    manual_seq('train', cb_rows)
])
Xva = np.column_stack([
    load_manual('valid', valid_rows)[manual_cols],
    manual_seq('valid', valid_rows)
])

manual_seq_search = tune('R2_ManualSeq', Xtr, Xva)
display(manual_seq_search.head())

model = CatBoostClassifier()
model.load_model(str(OUT / 'R2_ManualSeq_best.cbm'))

pred = np.empty(valid_store.n, np.float32)
offset = 0
for batch in tqdm(pq.ParquetFile(R2/'features'/'features_valid.parquet')
                  .iter_batches(batch_size=100_000, columns=manual_cols)):
    n = batch.num_rows
    r = slice(offset, offset + n)
    X = np.column_stack([
        batch.to_pandas()[manual_cols].to_numpy(np.float32),
        manual_seq('valid', r)
    ])
    pred[r] = model.predict_proba(X)[:, 1]
    offset += n


R2_ManualSeq:   0%|          | 0/10 [00:00<?, ?it/s]

0:	learn: 0.6345120	test: 0.6318426	best: 0.6318426 (0)	total: 36.9ms	remaining: 1m 32s
400:	learn: 0.0911412	test: 0.0413551	best: 0.0413551 (400)	total: 8.72s	remaining: 45.6s
800:	learn: 0.0854350	test: 0.0404987	best: 0.0404987 (800)	total: 17.4s	remaining: 36.9s
1200:	learn: 0.0813347	test: 0.0398648	best: 0.0398579 (1197)	total: 26.1s	remaining: 28.2s
1600:	learn: 0.0782033	test: 0.0395169	best: 0.0395169 (1600)	total: 34.8s	remaining: 19.6s
2000:	learn: 0.0755159	test: 0.0392000	best: 0.0392000 (2000)	total: 43.7s	remaining: 10.9s
2400:	learn: 0.0731195	test: 0.0388862	best: 0.0388862 (2400)	total: 52.5s	remaining: 2.17s
2499:	learn: 0.0725999	test: 0.0388361	best: 0.0388332 (2467)	total: 54.7s	remaining: 0us
bestTest = 0.0388332208
bestIteration = 2467
Shrink model to first 2468 iterations.
0:	learn: 0.6017328	test: 0.5968432	best: 0.5968432 (0)	total: 25.1ms	remaining: 1m 2s
400:	learn: 0.0927666	test: 0.0416373	best: 0.0416373 (400)	total: 6.56s	remaining: 34.3s
800:	learn: 0

,ndcg@20,params,trees
1,0.097298,"{'learning_rate': 0.05, 'l2_leaf_reg': 30, 'de...",2496
0,0.095088,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2468
3,0.094729,"{'learning_rate': 0.05, 'l2_leaf_reg': 30, 'de...",2497
2,0.093799,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",1444
6,0.093365,"{'learning_rate': 0.05, 'l2_leaf_reg': 10, 'de...",2498


0it [00:00, ?it/s]

In [51]:
scores['R2+ManualSeq'] = pred


## Сравнение

In [55]:
A = {s: np.load(LOCAL/f'learned_{s}.npy', mmap_mode='r') for s in ('train', 'valid')}
S = {s: np.load(LOCAL/f'seq_{s}.npy', mmap_mode='r') for s in ('train', 'valid')}

def seq_features(split, rows):
    st = globals()[f'{split}_store']
    return np.column_stack([A[split][rows], st.a['common'][rows], S[split][rows]])

seq_search = tune(
    'AutoFE_Seq',
    seq_features('train', cb_rows),
    seq_features('valid', valid_rows)
)
display(seq_search.head())


AutoFE_Seq:   0%|          | 0/10 [00:00<?, ?it/s]

0:	learn: 0.6322959	test: 0.6295272	best: 0.6295272 (0)	total: 40.5ms	remaining: 1m 41s
400:	learn: 0.0931118	test: 0.0417319	best: 0.0417313 (399)	total: 9.28s	remaining: 48.6s
800:	learn: 0.0891305	test: 0.0413865	best: 0.0413865 (800)	total: 18.5s	remaining: 39.3s
1200:	learn: 0.0861264	test: 0.0411648	best: 0.0411585 (1193)	total: 27.7s	remaining: 30s
1600:	learn: 0.0835215	test: 0.0410016	best: 0.0410016 (1600)	total: 37s	remaining: 20.8s
2000:	learn: 0.0812477	test: 0.0408655	best: 0.0408652 (1997)	total: 46.3s	remaining: 11.5s
2400:	learn: 0.0790685	test: 0.0407045	best: 0.0407044 (2399)	total: 55.6s	remaining: 2.29s
2499:	learn: 0.0785641	test: 0.0406723	best: 0.0406723 (2499)	total: 57.9s	remaining: 0us
bestTest = 0.04067227046
bestIteration = 2499
0:	learn: 0.5961409	test: 0.5917902	best: 0.5917902 (0)	total: 28.7ms	remaining: 1m 11s
400:	learn: 0.0948406	test: 0.0420788	best: 0.0420741 (383)	total: 7.2s	remaining: 37.7s
800:	learn: 0.0919809	test: 0.0417726	best: 0.0417715 (

,ndcg@20,params,trees
1,0.096714,"{'learning_rate': 0.05, 'l2_leaf_reg': 30, 'de...",2429
0,0.094945,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2500
6,0.094918,"{'learning_rate': 0.05, 'l2_leaf_reg': 10, 'de...",2422
9,0.094822,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'de...",2496
8,0.094557,"{'learning_rate': 0.08, 'l2_leaf_reg': 30, 'de...",1868


In [57]:
seq_cb = CatBoostClassifier(
    iterations=CB_TREES, depth=CB_DEPTH, learning_rate=CB_LR,
    loss_function='Logloss', random_seed=SEED,
    task_type='GPU' if DEVICE == 'cuda' else 'CPU', verbose=200
)
seq_cb.fit(seq_features('train', cb_rows), y_train,
           eval_set=(seq_features('valid', valid_rows), y_valid),
           early_stopping_rounds=120)

seq_scores = np.empty(valid_store.n, np.float32)
for i in tqdm(range(0, valid_store.n, 100_000)):
    r = slice(i, min(i+100_000, valid_store.n))
    seq_scores[r] = seq_cb.predict_proba(seq_features('valid', r))[:, 1]


0:	learn: 0.5942731	test: 0.5896982	best: 0.5896982 (0)	total: 41.5ms	remaining: 20.7s
200:	learn: 0.0940367	test: 0.0418447	best: 0.0418447 (200)	total: 4.75s	remaining: 7.07s
400:	learn: 0.0899828	test: 0.0415127	best: 0.0415089 (398)	total: 9.43s	remaining: 2.33s
499:	learn: 0.0884426	test: 0.0414036	best: 0.0414008 (498)	total: 11.7s	remaining: 0us
bestTest = 0.04140079816
bestIteration = 498
Shrink model to first 499 iterations.


  0%|          | 0/60 [00:00<?, ?it/s]

In [60]:
scores['AutoFE+Seq'] = seq_scores

comparison = pd.DataFrame([
    ranking_report(scores[k], valid_store, truth_valid, k)
    for k in ('R2', 'R2+ManualSeq', 'AutoFE', 'AutoFE+Seq')
])

display(comparison[['model', 'ndcg@10', 'ndcg@20', 'recall@20', 'logloss']])
comparison.to_csv(OUT / 'manual_seq_comparison.csv', index=False)


,model,ndcg@10,ndcg@20,recall@20,logloss
0,R2,0.094611,0.088879,0.064349,0.040061
1,R2+ManualSeq,0.093923,0.088704,0.064698,0.039568
2,AutoFE,0.093553,0.087792,0.064181,0.041338
3,AutoFE+Seq,0.094838,0.088924,0.065017,0.041586


## Интерпретация

In [61]:
w = 1 / np.log2(np.arange(2, 22))
ideal = np.r_[0, w.cumsum()]

def user_ndcg(pred, store, total):
    y = np.asarray(store.a['y']).reshape(N_USERS, C)
    top = np.argsort(-np.asarray(pred).reshape(N_USERS, C),
                     axis=1, kind='stable')[:, :20]
    dcg = (np.take_along_axis(y, top, axis=1) * w).sum(1)
    return np.where(total > 0, dcg / np.maximum(ideal[np.minimum(total, 20)], 1), 0)

v = {k: user_ndcg(scores[k], valid_store, truth_valid) for k in comparison.model}
rng_ci = np.random.default_rng(123)
ci = []

for a, b in [('AutoFE+Seq', 'R2'), ('AutoFE+Seq', 'AutoFE'),
             ('R2+ManualSeq', 'R2')]:
    d = v[a] - v[b]
    boot = [d[rng_ci.integers(N_USERS, size=N_USERS)].mean() for _ in range(2000)]
    ci.append((a, b, d.mean(), *np.quantile(boot, [.025, .975])))

display(pd.DataFrame(ci, columns=['model', 'baseline', 'delta_ndcg20', 'CI_low', 'CI_high']))

top = {k: np.argsort(-scores[k].reshape(N_USERS, C), axis=1)[:, :20]
       for k in ('R2', 'AutoFE+Seq')}
overlap = (top['R2'][:, :, None] == top['AutoFE+Seq'][:, None, :]).any(2).sum(1).mean() / 20
print(f'Top-20 overlap R2 vs AutoFE+Seq: {overlap:.1%}')

seq_names = learned_cols + raw_common + [f'seq_{i}' for i in range(S['valid'].shape[1]-1)] + ['seq_score']
imp = pd.Series(seq_cb.feature_importances_, index=seq_names).sort_values(ascending=False)
display(imp.head(20).rename('importance').to_frame())
print('Importance share of CandidateSeq:', round(imp[imp.index.str.startswith('seq_')].sum(), 2), '%')

files = {'R2 validation': R2/'features'/'features_valid.parquet',
         'AutoFE validation': LOCAL/'learned_valid.npy',
         'Seq validation': LOCAL/'seq_valid.npy',
         'AutoFE checkpoint': OUT/'encoder.pt',
         'Seq checkpoint': OUT/'seq_encoder.pt'}
display(pd.Series({k: round(p.stat().st_size / 2**20, 1) for k, p in files.items()}, name='MiB'))

print('Encoder parameters:', sum(p.numel() for p in encoder.parameters()),
      '| CandidateSeq parameters:', sum(p.numel() for p in seq.parameters()))


,model,baseline,delta_ndcg20,CI_low,CI_high
0,AutoFE+Seq,R2,0.000045,-0.001131,0.001251
1,AutoFE+Seq,AutoFE,0.001132,0.000452,0.001821
2,R2+ManualSeq,R2,-0.000175,-0.000807,0.000461


Top-20 overlap R2 vs AutoFE+Seq: 61.0%


,importance
itemknn_score,7.629637
decaypop_score,7.536373
latent_44,4.320953
source_is_itemknn,4.129888
latent_27,3.802414
latent_09,3.411149
latent_58,2.923957
latent_03,2.834576
latent_52,2.544461
latent_61,2.370290


Importance share of CandidateSeq: 0.23 %


,MiB
R2 validation,227.3
AutoFE validation,743.9
Seq validation,68.7
AutoFE checkpoint,0.4
Seq checkpoint,114.0


Encoder parameters: 107039 | CandidateSeq parameters: 29890349


## Тест

In [62]:
assert all(name in hist_id and name in hist_age for name in ('train', 'valid', 'test'))


In [63]:
neg_rng = np.random.default_rng(SEED + 1)
vy = np.asarray(valid_store.a['y']).reshape(N_USERS, C)
vneg = [u*C + neg_rng.choice(np.flatnonzero(vy[u] == 0),
        size=min(CB_NEG_PER_USER, (vy[u] == 0).sum()), replace=False) for u in range(N_USERS)]
vrows = np.sort(np.concatenate([np.flatnonzero(vy.ravel()), *vneg])).astype(np.int64)
y_final = np.r_[y_train, np.asarray(valid_store.a['y'][vrows], dtype=np.int8)]

def fit_features(split, rows, name):
    st = globals()[f'{split}_store']
    if name.startswith('R2'):
        x = load_manual(split, rows)[manual_cols].to_numpy(np.float32)
        return np.column_stack([x, manual_seq(split, rows)]) if name == 'R2+ManualSeq' else x
    return seq_features(split, rows) if name == 'AutoFE+Seq' else np.column_stack([
        A[split][rows], st.a['common'][rows]])

searches = {'R2': r2_search, 'R2+ManualSeq': manual_seq_search, 'AutoFE': a1_search}
final_models, training_seconds = {}, {}

for name in ('R2', 'R2+ManualSeq', 'AutoFE', 'AutoFE+Seq'):
    if name == 'AutoFE+Seq':
        p = seq_cb.get_all_params()
        hp, trees = {k: p[k] for k in ('depth', 'learning_rate', 'l2_leaf_reg')}, seq_cb.tree_count_
    else:
        best = searches[name].iloc[0]
        hp, trees = best['params'], int(best['trees'])

    X = np.vstack([fit_features('train', cb_rows, name), fit_features('valid', vrows, name)])
    t = time.perf_counter()
    model = CatBoostClassifier(**hp, iterations=trees, loss_function='Logloss',
        task_type='GPU' if DEVICE == 'cuda' else 'CPU', random_seed=SEED, verbose=200)
    model.fit(X, y_final)
    training_seconds[name] = round(time.perf_counter() - t, 1)
    model.save_model(str(OUT / f'{name.replace("+", "_")}_final.cbm'))
    final_models[name] = model
    print(name, len(X), 'rows,', trees, 'trees,', training_seconds[name], 's')
    del X
    gc.collect()

pd.Series(training_seconds, name='train_seconds').to_csv(OUT/'final_training_cost.csv')


0:	learn: 0.6373289	total: 46.1ms	remaining: 1m 54s
200:	learn: 0.1009245	total: 5.6s	remaining: 1m 4s
400:	learn: 0.0979915	total: 11s	remaining: 57.4s
600:	learn: 0.0961786	total: 16.5s	remaining: 52s
800:	learn: 0.0948938	total: 21.8s	remaining: 46.2s
1000:	learn: 0.0937834	total: 27.3s	remaining: 40.9s
1200:	learn: 0.0929130	total: 32.7s	remaining: 35.3s
1400:	learn: 0.0920975	total: 38.1s	remaining: 29.8s
1600:	learn: 0.0913650	total: 43.5s	remaining: 24.4s
1800:	learn: 0.0906812	total: 49s	remaining: 18.9s
2000:	learn: 0.0900834	total: 54.4s	remaining: 13.5s
2200:	learn: 0.0895234	total: 60s	remaining: 8.07s
2400:	learn: 0.0889747	total: 1m 5s	remaining: 2.62s
2496:	learn: 0.0887184	total: 1m 8s	remaining: 0us
R2 2074770 rows, 2497 trees, 72.4 s
0:	learn: 0.6022295	total: 52ms	remaining: 2m 9s
200:	learn: 0.0986046	total: 5.74s	remaining: 1m 5s
400:	learn: 0.0957713	total: 11.3s	remaining: 59.1s
600:	learn: 0.0938879	total: 16.8s	remaining: 53.1s
800:	learn: 0.0925108	total: 22.4

In [78]:
for name in ('train_store', 'valid_store', 'r2_train', 'r2_valid', 'a1_train',
             'a1_valid', 'Xtr', 'Xva', 'man_train', 'man_eval', 'mem_train',
             'mem_valid', 'manual', 'automatic', 'scores', 'seq_scores', 'pred'):
    globals().pop(name, None)
gc.collect()
ctypes.CDLL('libc.so.6').malloc_trim(0)
RAM_SHARE = 0.70

c_test = load_main_candidates('test')
cols_test = ['uid', 'item_id', 'group_id', 'label', 'retrieval_rank',
             'itemknn_score', 'decaypop_score', 'retrieval_source']
c_test = c_test[cols_test].copy()
gc.collect()
folder = LOCAL/'store_test'
if not store_ready('test', c_test, int(snaps.loc['test'].history_end)):
    shutil.rmtree(folder, ignore_errors=True)
    create_store('test', c_test, int(snaps.loc['test'].history_end))
del c_test
gc.collect()


test: compact observed mapping
test: indexing 23.6s, events=47,550,037, ui=2,279,851, ua=1,450,640
test: RAM staging 2.88 GiB / budget 3.00 GiB
test: event aggregation 908.3s
  ui: 2,279,852 rows, 5,646,992 nonempty time bins
  ua: 1,450,641 rows, 5,207,587 nonempty time bins
test: save and compression 55.7s
test history events 47,550,037 pairs 6,000,000 stored ui 2,279,851 stored ua 1,450,640 seconds 992.0


In [86]:
A = {s: np.load(LOCAL/f'learned_{s}.npy', mmap_mode='r')
     for s in ('train', 'valid')}
S = {s: np.load(LOCAL/f'seq_{s}.npy', mmap_mode='r')
     for s in ('train', 'valid')}


In [82]:
assert 'test' in hist_id and 'test' in hist_age


In [83]:
test_store = RelationStore('test')

if not feature_ready(LOCAL/'learned_test.npy', (test_store.n, Z_DIM+1), OUT/'encoder.pt', 'test'):
    encoder.load_state_dict(torch.load(OUT/'encoder.pt', map_location=DEVICE, weights_only=True))
    extract_features('test', test_store)

if not feature_ready(LOCAL/'seq_test.npy', (test_store.n, 6), OUT/'seq_encoder.pt', 'test'):
    seq.load_state_dict(torch.load(OUT/'seq_encoder.pt', map_location=DEVICE, weights_only=True))
    tmp = LOCAL/'seq_test.partial.npy'
    feat = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.float16,
                                    shape=(test_store.n, 6))
    for p in tqdm(range(0, test_store.n, 4096), desc='Seq test'):
        r = np.arange(p, min(p+4096, test_store.n))
        feat[p:p+len(r)] = seq_predict('test', test_store, r, features=True).astype(np.float16)
    feat.flush()
    del feat
    finish_feature(tmp, LOCAL/'seq_test.npy', (test_store.n, 6), OUT/'seq_encoder.pt', 'test')

A['test'] = np.load(LOCAL/'learned_test.npy', mmap_mode='r')
S['test'] = np.load(LOCAL/'seq_test.npy', mmap_mode='r')
assert_alignment('test', test_store)

sc = {name: np.empty(test_store.n, np.float32) for name in final_models}
a = np.load(LOCAL/'learned_test.npy', mmap_mode='r')
s = np.load(LOCAL/'seq_test.npy', mmap_mode='r')

for p in tqdm(range(0, test_store.n, 100_000), desc='AutoFE test'):
    r = slice(p, min(p+100_000, test_store.n))
    x = np.column_stack([a[r].astype(np.float32), test_store.a['common'][r]])
    sc['AutoFE'][r] = final_models['AutoFE'].predict_proba(x)[:, 1]
    sc['AutoFE+Seq'][r] = final_models['AutoFE+Seq'].predict_proba(np.column_stack([x, s[r]]))[:, 1]

p = 0
for b in tqdm(pq.ParquetFile(R2/'features'/'features_test.parquet').iter_batches(
             batch_size=100_000, columns=list(dict.fromkeys(manual_cols + ['uid', 'item_id', 'group_id']))), desc='Manual test'):
    f = b.to_pandas()
    x = f[manual_cols].to_numpy(np.float32)
    r = slice(p, p+len(x))
    sc['R2'][r] = final_models['R2'].predict_proba(x)[:, 1]
    sc['R2+ManualSeq'][r] = final_models['R2+ManualSeq'].predict_proba(
        np.column_stack([x, manual_seq('test', r)]))[:, 1]
    p += len(x)

assert p == test_store.n, 'Test row count mismatch'

truth_test = future_totals('test')
test_results = pd.DataFrame([ranking_report(sc[k], test_store, truth_test, k) for k in sc])
display(test_results[['model', 'ndcg@10', 'ndcg@20', 'recall@20', 'logloss']])
test_results.to_csv(OUT/'final_test_metrics.csv', index=False)

d = user_ndcg(sc['AutoFE+Seq'], test_store, truth_test) - user_ndcg(sc['R2'], test_store, truth_test)
rg = np.random.default_rng(123)
ci = np.quantile([d[rg.integers(N_USERS, size=N_USERS)].mean() for _ in range(2000)], [.025, .975])
print('Seq − R2 delta NDCG@20:', d.mean(), '95% bootstrap CI:', ci)

comparisons = []
for left, right in [('AutoFE', 'R2'), ('AutoFE+Seq', 'AutoFE'),
                    ('R2+ManualSeq', 'R2'), ('AutoFE+Seq', 'R2+ManualSeq')]:
    diff = user_ndcg(sc[left], test_store, truth_test) - user_ndcg(sc[right], test_store, truth_test)
    generator = np.random.default_rng(SEED + 2026)
    boot = [diff[generator.integers(N_USERS, size=N_USERS)].mean() for _ in range(2000)]
    comparisons.append(dict(model=left, baseline=right, delta_ndcg20=float(diff.mean()),
                            CI_low=float(np.quantile(boot, .025)),
                            CI_high=float(np.quantile(boot, .975))))
pd.DataFrame(comparisons).to_csv(OUT/'final_test_bootstrap.csv', index=False)


AutoFE test:   0%|          | 0/60 [00:00<?, ?it/s]

Manual test: 0it [00:00, ?it/s]

,model,ndcg@10,ndcg@20,recall@20,logloss
0,R2,0.093081,0.088039,0.064351,0.040463
1,R2+ManualSeq,0.093645,0.088626,0.064535,0.039895
2,AutoFE,0.093513,0.088317,0.064728,0.041310
3,AutoFE+Seq,0.093693,0.088379,0.064351,0.041567


Seq − R2 delta NDCG@20: 0.0003398964661963543 95% bootstrap CI: [-0.0007042   0.00140833]


## Инференс

In [88]:
from time import perf_counter

n = 50000
r = np.arange(n)

m = next(pq.ParquetFile(
    R2 / 'features' / 'features_test.parquet').iter_batches(batch_size=n)).to_pandas()[manual_cols].to_numpy(np.float32)

a = np.asarray(A['test'][r], dtype=np.float32)
s = np.asarray(S['test'][r], dtype=np.float32)
common = test_store.a['common'][r]

xs = {
    'R2': m,
    'R2+ManualSeq': np.column_stack([m, manual_seq('test', r)]),
    'AutoFE': np.column_stack([a, common]),
    'AutoFE+Seq': np.column_stack([a, common, s])
}

for name, x in xs.items():
    model = final_models[name]
    model.predict_proba(x)
    measurements = []
    for _ in range(7):
        t = perf_counter()
        model.predict_proba(x)
        measurements.append(perf_counter() - t)
    sec = float(np.median(measurements))
    print(f'{name}: median {sec:.3f} s / {n:,} rows, '
          f'{n / sec:,.0f} rows/s; trees={model.tree_count_}')


R2: 0.131 s / 50,000 rows, 382,042 rows/s
R2+ManualSeq: 0.155 s / 50,000 rows, 322,502 rows/s
AutoFE: 0.180 s / 50,000 rows, 278,406 rows/s
AutoFE+Seq: 0.073 s / 50,000 rows, 685,083 rows/s
